# 02 — DNA-Binding Protein Design

Held-out 10-bp dsDNA targets from PDB entries `7M5W`, `7RTE`, `7N5U` (RFD3 paper §3.3).

Per target:
1. Auto-detect DNA chains in the PDB and crop a 10-bp window
2. Generate ~12 backbones with each model
3. Sequence design with LigandMPNN

> **Caveat**: Chroma has no DNA support; results are illustrative only.
> RFD3-NA (`rfd3na`) is the dedicated NA-aware path; here we use base `rfd3 design`
> with DNA chains in the input.

In [ ]:
%cd /content/repo
import sys
if '/content/repo/scripts' not in sys.path:
    sys.path.insert(0, '/content/repo/scripts')

from utils import (RESULTS, DATA, RunRecord, append_record, fetch_pdb,
                   free_gpu, rfd3_run, run_ligandmpnn, ensure_ligandmpnn)
import numpy as np, time, json, os
from pathlib import Path
import biotite.structure.io.pdb as bpdb

DNA_TARGETS = ['7m5w', '7rte', '7n5u']
N_DESIGNS = 8
LEN_RANGE = (50, 80)
DNA_RESNAMES = {'DA', 'DT', 'DG', 'DC'}

In [ ]:
def auto_dna_window(pdb_id, n_bp=10):
    arr = bpdb.PDBFile.read(fetch_pdb(pdb_id)).get_structure(model=1)
    dna_mask = np.isin(arr.res_name, list(DNA_RESNAMES))
    if not dna_mask.any():
        raise RuntimeError(f'{pdb_id}: no DNA atoms found')
    dna = arr[dna_mask]
    chains = list(np.unique(dna.chain_id))[:2]   # take first two strands
    keep = np.zeros(len(arr), dtype=bool)
    chain_resid = {}
    for ch in chains:
        ch_mask = (arr.chain_id == ch) & dna_mask
        ch_resids = sorted(set(arr.res_id[ch_mask]))[:n_bp]
        keep |= ch_mask & np.isin(arr.res_id, ch_resids)
        chain_resid[ch] = ch_resids
    sub = arr[keep]
    out = DATA / 'dna_targets' / f'{pdb_id}_dna.pdb'
    out.parent.mkdir(parents=True, exist_ok=True)
    f = bpdb.PDBFile(); f.set_structure(sub); f.write(out)
    return out, chain_resid

dna_targets = {}
for pid in DNA_TARGETS:
    try:
        path, chain_resid = auto_dna_window(pid)
        dna_targets[pid] = {'pdb': path, 'chain_resid': chain_resid}
        print(f'{pid}: chains={list(chain_resid)}, file={path.name}')
    except Exception as e:
        print(f'{pid}: SKIP ({e})')

## RFdiffusion3 — DNA conditional generation

In [ ]:
rfd3_dna = DATA / 'rfd3_dna'
rfd3_dna.mkdir(exist_ok=True)

def dna_contig(chain_resid, binder_len):
    """Build a contig string: <binder_len>,/0,<DNA chain ranges>"""
    parts = [str(binder_len), '/0']
    for ch, resids in chain_resid.items():
        if resids:
            parts.append(f'{ch}{resids[0]}-{resids[-1]}')
    return ','.join(parts)

for pid, info in dna_targets.items():
    L = LEN_RANGE[1]
    spec = {
        f'dna_{pid}': {
            'input': str(info['pdb']),
            'contig': dna_contig(info['chain_resid'], L),
            'length': f'{LEN_RANGE[0]}-{L}',
        }
    }
    out_dir = rfd3_dna / pid
    ok, err, dt = rfd3_run(spec, out_dir,
                            diffusion_batch_size=N_DESIGNS,
                            num_timesteps=200)
    if not ok:
        print(f'RFD3 {pid}: FAILED — {err[-200:]}')
        continue
    append_record(RunRecord(
        model='rfd3', task='dna_binder', target=pid, length=L,
        n_designs=N_DESIGNS, seconds=dt,
        metrics={'s_per_design': dt / N_DESIGNS},
    ))
    print(f'RFD3 {pid}: {dt/N_DESIGNS:.1f}s/design')

free_gpu()

## Chroma — `ProClassConditioner` baseline

Chroma cannot model DNA. We use class-conditional sampling toward CATH "α/β"
(class 3) — many DNA-binding folds are in this class. Strict baseline only.

In [ ]:
from chroma import Chroma, api, conditioners
api.register_key(os.environ['CHROMA_API_KEY'])
chroma = Chroma()

chroma_dna = DATA / 'chroma_dna'
chroma_dna.mkdir(exist_ok=True)

for pid in dna_targets:
    times = []
    for i in range(N_DESIGNS):
        L = int(np.random.randint(*LEN_RANGE))
        try:
            cond = conditioners.ProClassConditioner(
                label='cath', value='3', model='named:public', weight=5.0
            )
            t0 = time.perf_counter()
            protein = chroma.sample(chain_lengths=[L], steps=200,
                                    conditioner=cond, sde_func='langevin')
            times.append(time.perf_counter() - t0)
            protein.to(str(chroma_dna / f'{pid}_n{i:02d}.pdb'))
        except Exception as e:
            print(f'Chroma {pid} #{i}: {type(e).__name__}: {e}')
    if times:
        append_record(RunRecord(
            model='chroma', task='dna_binder', target=pid, length=LEN_RANGE[1],
            n_designs=len(times), seconds=sum(times),
            metrics={'s_per_design': float(np.mean(times))},
        ))
        print(f'Chroma {pid}: {np.mean(times):.1f}s/design (n={len(times)})')

free_gpu()

## LigandMPNN sequence design (sanity)

Detailed refolding evaluation lives in notebook 05. Here we just verify the
pipeline works on a few designs.

In [ ]:
ensure_ligandmpnn()    # idempotent

designs_with_seqs = {}
for tag, root in [('rfd3', rfd3_dna), ('chroma', chroma_dna)]:
    for pid in dna_targets:
        files = sorted(root.rglob(f'*{pid}*.pdb'))[:3]
        for f in files:
            ok = run_ligandmpnn(f, f.parent / 'mpnn' / f.stem,
                                model_type='ligand_mpnn',
                                checkpoint='ligandmpnn_v_32_010_25.pt')
            if ok:
                designs_with_seqs.setdefault((tag, pid), []).append(f)

print({f'{k[0]}_{k[1]}': len(v) for k, v in designs_with_seqs.items()})

In [ ]:
summary = {f'{tag}_{pid}': {'n_designs': len(v)}
           for (tag, pid), v in designs_with_seqs.items()}
(RESULTS / 'dna_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))